# 02 — NLP Text Preprocessing & TF-IDF Feature Extraction
**Project:** Fake News Detection Using Natural Language Processing

This notebook demonstrates the exact preprocessing pipeline shared across training (`train_model.py`) and live inference (`backend/app.py` via `backend/model_utils.py`):
1. Combining `title` and `text` (`content = title + " " + text`)
2. Converting text to lowercase
3. Removing URLs (`http://`, `https://`, `www.`)
4. Removing HTML tags (`<...>`)
5. Removing non-alphabetic characters (numbers, punctuation, special symbols)
6. Removing extra whitespace and stripping leading/trailing spaces
7. Transforming cleaned text into TF-IDF unigram and bigram features

## 1. Import Shared Preprocessing Utilities

In [1]:
import re
import sys
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.model_utils import clean_text, combine_title_and_text

## 2. Step-by-Step Demonstration Before & After Preprocessing

In [2]:
sample_title = "BREAKING NEWS: Senate Passes $1.2 Trillion Bill! Read more at https://example.com/news"
sample_text = (
    "<p>WASHINGTON (Reuters) - Lawmakers voted <b>69-30</b> on Tuesday! "
    "Visit www.senate.gov/bill_2026 for full details...   Multiple agencies   confirmed the report.</p>"
)

# Step 0: Combine title and text
combined = combine_title_and_text(sample_title, sample_text)

# Step 1: Lowercase
step1_lower = combined.lower()

# Step 2: Remove URLs
step2_no_urls = re.sub(r"https?://\S+|www\.\S+", " ", step1_lower)

# Step 3: Remove HTML tags
step3_no_html = re.sub(r"<.*?>+", " ", step2_no_urls)

# Step 4: Remove non-alphabetic characters
step4_alpha_only = re.sub(r"[^a-z\s]", " ", step3_no_html)

# Step 5 & 6: Normalize whitespace and strip
step5_normalized = re.sub(r"\s+", " ", step4_alpha_only).strip()

print("=== BEFORE PREPROCESSING (COMBINED) ===")
print(combined)
print("\n=== STEP 1: LOWERCASE ===")
print(step1_lower)
print("\n=== STEP 2: URL REMOVAL ===")
print(step2_no_urls)
print("\n=== STEP 3: HTML TAG REMOVAL ===")
print(step3_no_html)
print("\n=== STEP 4: NON-ALPHABETIC REMOVAL ===")
print(step4_alpha_only)
print("\n=== STEP 5 & 6: WHITESPACE NORMALIZATION (FINAL) ===")
print(step5_normalized)

# Verify identical output with shared clean_text() function
assert step5_normalized == clean_text(combined)
print("\nVerified: Matches backend.model_utils.clean_text() identically!")

=== BEFORE PREPROCESSING (COMBINED) ===
BREAKING NEWS: Senate Passes $1.2 Trillion Bill! Read more at https://example.com/news <p>WASHINGTON (Reuters) - Lawmakers voted <b>69-30</b> on Tuesday! Visit www.senate.gov/bill_2026 for full details...   Multiple agencies   confirmed the report.</p>

=== STEP 1: LOWERCASE ===
breaking news: senate passes $1.2 trillion bill! read more at https://example.com/news <p>washington (reuters) - lawmakers voted <b>69-30</b> on tuesday! visit www.senate.gov/bill_2026 for full details...   multiple agencies   confirmed the report.</p>

=== STEP 2: URL REMOVAL ===
breaking news: senate passes $1.2 trillion bill! read more at   <p>washington (reuters) - lawmakers voted <b>69-30</b> on tuesday! visit   for full details...   multiple agencies   confirmed the report.</p>

=== STEP 3: HTML TAG REMOVAL ===
breaking news: senate passes $1.2 trillion bill! read more at    washington (reuters) - lawmakers voted  69-30  on tuesday! visit   for full details...   mul

## 3. Apply Preprocessing on Sample Corpus & Inspect TF-IDF Features

In [4]:
DATASET_DIR = PROJECT_ROOT / "dataset"
FAKE_PATH = DATASET_DIR / "Fake.csv"
TRUE_PATH = DATASET_DIR / "True.csv"

if FAKE_PATH.is_file() and TRUE_PATH.is_file():
    fake_df = pd.read_csv(FAKE_PATH).head(500)
    true_df = pd.read_csv(TRUE_PATH).head(500)
    fake_df["label"] = 0
    true_df["label"] = 1
    sample_df = pd.concat([fake_df, true_df], ignore_index=True)
    sample_df["raw_content"] = sample_df["title"].fillna("") + " " + sample_df["text"].fillna("")
    sample_df["cleaned_content"] = sample_df["raw_content"].apply(clean_text)
    display(sample_df[["raw_content", "cleaned_content", "label"]].head())
else:
    print("Dataset CSV files not yet placed in dataset/. Showing TF-IDF demonstration on synthetic examples:")
    demo_docs = [
        clean_text("Senate passes bipartisan infrastructure legislation after Washington debate"),
        clean_text("Federal Reserve announces benchmark interest rate policy decision in Washington"),
        clean_text("Shocking secret conspiracy revealed by anonymous internet whistleblower!"),
        clean_text("Miracle cure hidden by mainstream media exposed in viral video!"),
    ]
    demo_vectorizer = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        stop_words="english",
    )
    tfidf_matrix = demo_vectorizer.fit_transform(demo_docs)
    print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
    print("Sample Extracted Unigrams & Bigrams:", list(demo_vectorizer.get_feature_names_out())[:20])

,raw_content,cleaned_content,label
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump sends out embarrassing new year s...,0
1,Drunk Bragging Trump Staffer Started Russian ...,drunk bragging trump staffer started russian c...,0
2,Sheriff David Clarke Becomes An Internet Joke...,sheriff david clarke becomes an internet joke ...,0
3,Trump Is So Obsessed He Even Has Obama’s Name...,trump is so obsessed he even has obama s name ...,0
4,Pope Francis Just Called Out Donald Trump Dur...,pope francis just called out donald trump duri...,0
